In [ ]:
# ==========================================================
# Import Required Libraries
# ==========================================================
# This project uses pandas for data manipulation, NumPy for
# numerical operations, scikit-learn for machine learning,
# and matplotlib for result visualization.

import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import os

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import RepeatedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

# ==========================================================
# Load Dataset
# ==========================================================
# Read the original dataset and remove unnecessary columns
# that are not required for model training.

main_data_path = r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\02. data\Full data.csv"

df = pd.read_csv(main_data_path)

columns_drop = [0, 1, 71, 72, 73, 74]
df.drop(df.columns[columns_drop], axis=1, inplace=True)

print(df.head())

# ==========================================================
# Helper Function
# ==========================================================
# Extract numeric values from strings containing units
# (e.g., "12 mg/L" -> 12.0). Non-numeric values become NaN.

def clean_numeric_with_units(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float)):
        return x

    match = re.search(r"-?\d+\.?\d*", str(x))

    if match:
        return float(match.group())

    return np.nan

# ==========================================================
# Convert Multi-label Outputs into Binary Columns
# ==========================================================
# Each pictogram label is converted into an individual binary
# target column (One-vs-Rest representation).

df["Pictogram(s)"] = (
    df["Pictogram(s)"]
    .astype(str)
    .str.replace(r"[\[\]']", "", regex=True)
    .str.split(",")
    .apply(lambda x: [i.strip() for i in x if i.strip()])
)

all_outputs = sorted(
    set(item.strip() for sublist in df["Pictogram(s)"] for item in sublist)
)

for output in all_outputs:
    df[output] = df["Pictogram(s)"].apply(lambda x: int(output in x))

output = df[list(all_outputs)].values

# ==========================================================
# Feature Selection
# ==========================================================
# Remove the target columns and keep only predictor features.

feature_cols = [
    col for col in df.columns
    if col not in list(all_outputs) + ["Pictogram(s)"]
]

features = df[feature_cols].copy()

# ==========================================================
# Clean Numeric Features
# ==========================================================
# Convert numeric values stored as strings into actual
# numerical values whenever possible.

for col in features.columns:
    if features[col].dtype == object:
        cleaned = features[col].apply(clean_numeric_with_units)

        if cleaned.notna().sum() > 0:
            features[col] = cleaned

# ==========================================================
# Handle Missing Values
# ==========================================================
# Missing numerical values are replaced with the median,
# while categorical values are replaced with the most
# frequently occurring category.

numeric_cols = features.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_cols = features.select_dtypes(
    include=["object"]
).columns

num_imputer = SimpleImputer(strategy="median")
features[numeric_cols] = num_imputer.fit_transform(features[numeric_cols])

cat_imputer = SimpleImputer(strategy="most_frequent")
features[categorical_cols] = cat_imputer.fit_transform(features[categorical_cols])

# ==========================================================
# Encode Categorical Variables
# ==========================================================
# Convert categorical text features into integer labels.

le_dict = {}

for col in categorical_cols:
    features[col] = features[col].astype(str)

    le = LabelEncoder()
    features[col] = le.fit_transform(features[col])

    le_dict[col] = le

# ==========================================================
# Model Evaluation Setup
# ==========================================================
# Use Repeated K-Fold Cross Validation to obtain more stable
# performance estimates.

n_splits = 3
n_repeats = 50

rkf = RepeatedKFold(
    n_splits=n_splits,
    n_repeats=n_repeats,
    random_state=42
)

results = []
all_outputs = sorted(list(all_outputs))

# ==========================================================
# Train One Binary Classifier for Each Pictogram
# ==========================================================
# A separate Random Forest classifier is trained for every
# output class. Accuracy and F1-score are recorded for each
# fold.

for fold_counter, (train_idx, test_idx) in enumerate(
        rkf.split(features), start=1):

    X_train = features.iloc[train_idx]
    X_test = features.iloc[test_idx]

    k = fold_counter % n_splits

    if k == 0:
        k = n_splits

    n = (fold_counter - 1) // n_splits + 1

    for class_name in all_outputs:

        y = df[class_name].values

        y_train = y[train_idx]
        y_test = y[test_idx]

        clf = RandomForestClassifier(
            n_estimators=100,
            random_state=42
        )

        clf.fit(X_train, y_train)

        y_pred = clf.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, zero_division=0)

        results.append({
            "k": k,
            "n": n,
            "output": class_name,
            "accuracy": acc,
            "f1score": f1
        })

# ==========================================================
# Save Evaluation Results
# ==========================================================
# Store all fold results into a CSV file.

results_df = pd.DataFrame(results)

result_csv_path = r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\imbalanced data\result_imbalanced.csv"

results_df.to_csv(result_csv_path, index=False)

print("Results saved!")

# ==========================================================
# Generate F1-score Bar Charts
# ==========================================================
# Create one bar chart for each output class showing the
# F1-score obtained across all validation folds.

f1score_df = pd.read_csv(r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\imbalanced data\result_imbalanced.csv")

save_dir = r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\imbalanced data\plots_f1"

os.makedirs(save_dir, exist_ok=True)

outputs = f1score_df["output"].unique()

for output_name in outputs:

    data = f1score_df[
        f1score_df["output"] == output_name
    ].reset_index(drop=True)

    plt.figure(figsize=(12, 4))

    plt.bar(
        range(len(data)),
        data["f1score"],
        color="skyblue"
    )

    plt.title(f"Output: {output_name}")

    plt.ylabel("F1-score")

    plt.ylim(0, 1)

    plt.xticks([])

    save_path = os.path.join(
        save_dir,
        f"{output_name}_f1_plot.png"
    )

    plt.savefig(save_path, bbox_inches="tight")

    plt.close()

print("All plots saved.")

# ==========================================================
# Calculate Average F1-score
# ==========================================================
# Compute the average F1-score for each output class.

f1_df = pd.read_csv(r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\imbalanced data\result_imbalanced.csv")

average_f1_df = (
    f1_df.groupby("output")["f1score"]
    .mean()
    .reset_index()
)

average_f1_df.rename(
    columns={"f1score": "average_f1score"},
    inplace=True
)

average_f1_df.to_csv(
    r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\imbalanced data\average_f1score.csv",
    index=False
)

print("Average F1-scores saved.")

# ==========================================================
# Calculate Average Accuracy
# ==========================================================
# Compute the average accuracy for each output class.

accuracy_df = pd.read_csv(r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\imbalanced data\result_imbalanced.csv")

average_accuracy_df = (
    accuracy_df.groupby("output")["accuracy"]
    .mean()
    .reset_index()
)

average_accuracy_df.rename(
    columns={"accuracy": "average_accuracy"},
    inplace=True
)

average_accuracy_df.to_csv(
    r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\imbalanced data\average_accuracy.csv",
    index=False
)

print("Average accuracy saved.")